# Simple Time series Analysis on my portfolio
This is a basic time series EDA and basic portfolio analysis for my first portfolio (currently losing money xd)

# 1. Libs and config

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# 2. Data

In [2]:
# historical transacons
transactions = pd.DataFrame(
    [
        ("2026-09-08", None, "deposit", 1067.00),
        ("2026-09-09", "VT", "buy", -500.47),
        ("2026-09-10", "NVDA", "buy", -53.50),
        ("2026-09-10", "GOOGL", "buy", -53.50),
        ("2026-09-10", "AMZN", "buy", -80.15),
        ("2026-09-10", "XLV", "buy", -133.50),
        ("2026-09-10", "FIDU", "buy", -133.50),
        ("2026-09-10", "VT", "buy", -32.23),
        ("2026-09-10", "TSM", "buy", -80.15),
        ("2026-09-23", "VT", "dividend", 0.95),
        ("2026-09-23", "FIDU", "dividend", 0.21),
        ("2026-09-24", "XLV", "dividend", 0.36),
        ("2026-10-09", "TSM", "dividend", 0.16),
    ],
    columns=["date", "ticker", "type", "amount"],
)

transactions["date"] = pd.to_datetime(transactions["date"])

transactions

,date,ticker,type,amount
0,2026-09-08,NaN,deposit,1067.00
1,2026-09-09,VT,buy,-500.47
2,2026-09-10,NVDA,buy,-53.50
3,2026-09-10,GOOGL,buy,-53.50
4,2026-09-10,AMZN,buy,-80.15
5,2026-09-10,XLV,buy,-133.50
6,2026-09-10,FIDU,buy,-133.50
7,2026-09-10,VT,buy,-32.23
8,2026-09-10,TSM,buy,-80.15
9,2026-09-23,VT,dividend,0.95


In [3]:
# current holding

holdings = pd.DataFrame(
    [
        ("VT", 3.32195, 492.57, -39.87, -0.0748),
        ("XLV", 0.80476, 148.01, 14.67, 0.1100),
        ("FIDU", 1.46780, 126.68, -6.66, -0.0499),
        ("AMZN", 0.31714, 78.25, -1.74, -0.0218),
        ("TSM", 0.18514, 64.31, -15.68, -0.1960),
        ("GOOGL", 0.16142, 44.38, -8.96, -0.1680),
        ("NVDA", 0.24438, 36.96, -16.38, -0.3071),
    ],
    columns=["ticker", "quantity", "market_value", "reported_pnl", "reported_return"],
)

holdings["current_price"] = holdings["market_value"] / holdings["quantity"]

holdings

,ticker,quantity,market_value,reported_pnl,reported_return,current_price
0,VT,3.32195,492.57,-39.87,-0.0748,148.277367
1,XLV,0.80476,148.01,14.67,0.1100,183.918187
2,FIDU,1.46780,126.68,-6.66,-0.0499,86.306036
3,AMZN,0.31714,78.25,-1.74,-0.0218,246.736457
4,TSM,0.18514,64.31,-15.68,-0.1960,347.358756
5,GOOGL,0.16142,44.38,-8.96,-0.1680,274.934952
6,NVDA,0.24438,36.96,-16.38,-0.3071,151.239872


In [23]:
# Downloading historical data
tickers = holdings["ticker"].unique()

data = {}

for ticker in tickers:
    asset = yf.Ticker(ticker)

    asset_history = asset.history(
        period="10y",
        interval="1d",
        auto_adjust=False,
    )

    data[ticker] = asset_history

# S&P data
sp = yf.Ticker("SPY")
sp_history = sp.history(period="10y", interval="1d", auto_adjust=False)

# FRED data for risk free rate proxy (daily)

series = "DGS3MO"
url = "https://fred.stlouisfed.org/graph/" f"fredgraph.csv?id={series}"
rf = pd.read_csv(url, index_col=0, parse_dates=True)
display(rf.tail())

,DGS3MO
observation_date,
2026-11-18,4.34
2026-11-19,4.34
2026-11-20,4.32
2026-11-23,4.37
2026-11-24,4.36


In [24]:
data["NVDA"]

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2016-11-28 00:00:00-05:00,2.346750,2.368500,2.340000,2.352750,2.312243,278736000,0.0,0.0
2016-11-29 00:00:00-05:00,2.350000,2.361000,2.313250,2.331250,2.291113,444568000,0.0,0.0
2016-11-30 00:00:00-05:00,2.340000,2.354000,2.302500,2.305000,2.265315,440500000,0.0,0.0
2016-12-01 00:00:00-05:00,2.302500,2.304250,2.119250,2.191000,2.153277,1035524000,0.0,0.0
2016-12-02 00:00:00-05:00,2.156250,2.220000,2.128000,2.211250,2.173180,478292000,0.0,0.0
...,...,...,...,...,...,...,...,...
2026-11-19 00:00:00-05:00,152.919998,158.130005,145.710007,145.979996,145.979996,322940100,0.0,0.0
2026-11-20 00:00:00-05:00,143.520004,143.729996,124.500000,125.529999,125.529999,633418400,0.0,0.0
2026-11-23 00:00:00-05:00,167.039993,171.500000,156.929993,165.050003,165.050003,782593300,0.0,0.0


In [26]:
prices = pd.DataFrame({ticker: data[ticker]["Adj Close"] for ticker in tickers})

benchmark = pd.DataFrame(
    {"VT": data["VT"]["Adj Close"], "SPY": sp_history["Adj Close"]}
)
display(prices.head())
display(benchmark.head())

,VT,XLV,FIDU,AMZN,TSM,GOOGL,NVDA
Date,,,,,,,
2016-11-28 00:00:00-05:00,48.980694,58.333828,27.824659,38.338501,23.517252,38.919151,2.312243
2016-11-29 00:00:00-05:00,49.078133,58.714470,27.807394,38.125999,23.430033,39.099937,2.291113
2016-11-30 00:00:00-05:00,49.005035,58.156158,27.885069,37.528500,23.541046,38.428326,2.265315
2016-12-01 00:00:00-05:00,48.785847,57.733250,28.040415,37.182499,22.637142,37.856277,2.153277
2016-12-02 00:00:00-05:00,48.891396,57.868580,27.988632,37.016998,22.882946,37.862705,2.173180


,VT,SPY
Date,,
2016-11-28 00:00:00-05:00,48.980694,188.288910
2016-11-29 00:00:00-05:00,49.078133,188.656143
2016-11-30 00:00:00-05:00,49.005035,188.203506
2016-12-01 00:00:00-05:00,48.785847,187.511749
2016-12-02 00:00:00-05:00,48.891396,187.605743


In [28]:
# checking some general stats

display(prices.isna().sum())
display(benchmark.isna().sum())

print(f"Min date: {prices.index.min():%Y-%m-%d}")
print(f"Max date: {prices.index.max():%Y-%m-%d}")

VT       0
XLV      0
FIDU     0
AMZN     0
TSM      0
GOOGL    0
NVDA     0
dtype: int64

VT     0
SPY    0
dtype: int64

Min date: 2016-11-28
Max date: 2026-11-25


In [22]:
sp_history

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2016-11-28 00:00:00-05:00,2210.209961,2211.139893,2200.360107,2201.719971,2201.719971,3515100000,0.0,0.0
2016-11-29 00:00:00-05:00,2200.760010,2210.459961,2198.149902,2204.659912,2204.659912,3709190000,0.0,0.0
2016-11-30 00:00:00-05:00,2204.969971,2214.100098,2198.810059,2198.810059,2198.810059,5538400000,0.0,0.0
2016-12-01 00:00:00-05:00,2200.169922,2202.600098,2187.439941,2191.080078,2191.080078,5067470000,0.0,0.0
2016-12-02 00:00:00-05:00,2191.120117,2197.949951,2188.370117,2191.949951,2191.949951,3782370000,0.0,0.0
...,...,...,...,...,...,...,...,...
2026-11-19 00:00:00-05:00,7389.259766,7415.640137,7173.629883,7175.830078,7175.830078,5692800000,0.0,0.0
2026-11-20 00:00:00-05:00,7150.390137,7198.669922,6882.660156,6910.390137,6910.390137,6679680000,0.0,0.0
2026-11-23 00:00:00-05:00,7227.410156,7295.600098,7152.660156,7237.169922,7237.169922,5608530000,0.0,0.0


In [29]:
holdings

,ticker,quantity,market_value,reported_pnl,reported_return,current_price
0,VT,3.32195,492.57,-39.87,-0.0748,148.277367
1,XLV,0.80476,148.01,14.67,0.1100,183.918187
2,FIDU,1.46780,126.68,-6.66,-0.0499,86.306036
3,AMZN,0.31714,78.25,-1.74,-0.0218,246.736457
4,TSM,0.18514,64.31,-15.68,-0.1960,347.358756
5,GOOGL,0.16142,44.38,-8.96,-0.1680,274.934952
6,NVDA,0.24438,36.96,-16.38,-0.3071,151.239872


In [41]:
close_prices = pd.DataFrame({ticker: data[ticker]["Close"] for ticker in tickers})

portfolio_prices = close_prices.loc["2026-09-10":]

quantities = holdings.set_index("ticker")["quantity"]
print(quantities)
display(portfolio_prices.head())

ticker
VT       3.32195
XLV      0.80476
FIDU     1.46780
AMZN     0.31714
TSM      0.18514
GOOGL    0.16142
NVDA     0.24438
Name: quantity, dtype: float64


,VT,XLV,FIDU,AMZN,TSM,GOOGL,NVDA
Date,,,,,,,
2026-09-10 00:00:00-04:00,158.529999,165.660004,90.540001,251.889999,428.029999,332.600006,218.360001
2026-09-11 00:00:00-04:00,159.940002,165.360001,91.580002,256.779999,433.239990,338.500000,218.289993
2026-09-14 00:00:00-04:00,158.729996,167.750000,90.230003,253.539993,418.010010,349.390015,210.960007
2026-09-15 00:00:00-04:00,158.029999,167.660004,89.580002,248.419998,413.750000,344.980011,212.169998
2026-09-16 00:00:00-04:00,157.410004,167.770004,89.559998,245.960007,417.720001,342.869995,213.899994


# 3. Questions

## 3.1. How has my portfolio evolved since inception? 

In [ ]:
market_value = portfolio_prices * quantities
market_value["total"] = market_value.sum(axis=1)

market_value.tail()

,VT,XLV,FIDU,AMZN,TSM,GOOGL,NVDA,total
Date,,,,,,,,
2026-11-19 00:00:00-05:00,488.891375,142.812715,125.438187,81.622320,65.185942,44.012778,35.674591,983.637908
2026-11-20 00:00:00-05:00,471.982662,142.635667,121.372386,78.276497,59.005968,42.823113,30.677021,946.773314
2026-11-23 00:00:00-05:00,492.346232,144.390038,126.289513,80.994384,67.316905,45.536583,40.334920,997.208575
2026-11-24 00:00:00-05:00,490.186934,148.655268,125.306090,80.011248,64.650890,45.044249,37.971766,991.826445
2026-11-25 00:00:00-05:00,490.120481,147.818311,126.685814,77.518528,63.380827,44.083803,36.485935,986.093699


In [60]:
# Aggregate dividends by payment date
dividends = transactions.query("type == 'dividend'").groupby("date")["amount"].sum()

if market_value.index.tz is not None:
    dividends.index = dividends.index.tz_localize(market_value.index.tz)

# consturc cumulative dividend cash
cumulative_dividends = (
    dividends.reindex(market_value.index.union(dividends.index), fill_value=0)
    .sort_index()
    .cumsum()
    .reindex(market_value.index)
)

# Reconstruct total portfolio wealth
portfolio_wealth = market_value["total"] + cumulative_dividends
portfolio_returns = portfolio_wealth.pct_change().dropna()
portfolio_wealth.tail()

Date
2026-11-19 00:00:00-05:00    985.317908
2026-11-20 00:00:00-05:00    948.453314
2026-11-23 00:00:00-05:00    998.888575
2026-11-24 00:00:00-05:00    993.506445
2026-11-25 00:00:00-05:00    987.773699
dtype: float64

Date
2026-11-05   -0.043314
Freq: B, dtype: float64

In [ ]:
# no dividends reinvested (f hapi)

TRADING_DAYS = 252

initial_investment = transactions.loc[transactions["type"] == "deposit", "amount"].sum()

final_securities_value = market_value["total"].iloc[-1]
total_dividends = cumulative_dividends.iloc[-1]

w_t = portfolio_wealth.iloc[-1]
net_pnl = w_t - initial_investment

# Holding-period return since initial deposit
hpr = w_t / initial_investment - 1


portfolio_returns = portfolio_wealth.pct_change().dropna()

arithmetic_mean = portfolio_returns.mean()

geometric_mean = np.prod(1 + portfolio_returns) ** (1 / len(portfolio_returns)) - 1

annualized_mean = arithmetic_mean * TRADING_DAYS


std = portfolio_returns.std()
annualized_std = std * np.sqrt(TRADING_DAYS)

running_max = portfolio_wealth.cummax()

drawdown = portfolio_wealth / running_max - 1

maximum_drawdown = drawdown.min()
worst_daily_return = portfolio_returns.min()


# Parametric VaR (normal distribution)
parametric_var_95 = norm.ppf(0.05, arithmetic_mean, std)
parametric_var_99 = norm.ppf(0.01, arithmetic_mean, std)

# Nonparametric VaR (historical distribution)
historical_var_95 = portfolio_returns.quantile(0.05)
historical_var_99 = portfolio_returns.quantile(0.01)

# Historical expected shortfall
expected_shortfall_95 = portfolio_returns[portfolio_returns <= historical_var_95].mean()

expected_shortfall_99 = portfolio_returns[portfolio_returns <= historical_var_99].mean()


# Normalize datetime indices
portfolio_returns.index = (
    pd.to_datetime(portfolio_returns.index).tz_localize(None).normalize()
)

rf.index = pd.to_datetime(rf.index).tz_localize(None).normalize()

# Convert annualized percentage yields into daily returns
rf["daily_rf"] = (1 + rf["DGS3MO"] / 100) ** (1 / TRADING_DAYS) - 1

# Align risk-free rates with portfolio valuation dates
rf_aligned = (
    rf["daily_rf"]
    .sort_index()
    .dropna()
    .reindex(portfolio_wealth.index.tz_localize(None).normalize(), method="ffill")
    .shift(1)
)

# Construct excess return series
excess_return = (portfolio_returns - rf_aligned).dropna()

excess_return_mean = excess_return.mean()
excess_return_std = excess_return.std()


sharpe_ratio = excess_return_mean / excess_return_std

annualized_sharpe = sharpe_ratio * np.sqrt(TRADING_DAYS)

In [89]:
print("=" * 65)
print("          PERSONAL PORTFOLIO PERFORMANCE REPORT")
print("=" * 65)

print("\nPORTFOLIO VALUATION")
print("-" * 65)

print(f"Initial investment:          ${initial_investment:>12,.2f}")
print(f"Final securities value:      ${final_securities_value:>12,.2f}")
print(f"Accumulated dividends:       ${total_dividends:>12,.2f}")
print(f"Total portfolio wealth:      ${w_t:>12,.2f}")
print(f"Net profit/loss:             ${net_pnl:>12,.2f}")

print("\nRETURN METRICS")
print("-" * 65)

print(f"Holding-period return:        {hpr:>12.2%}")
print(f"Daily arithmetic mean:        {arithmetic_mean:>12.4%}")
print(f"Daily geometric mean:         {geometric_mean:>12.4%}")
print(f"Annualized arithmetic mean:   {annualized_mean:>12.2%}")

print("\nVOLATILITY METRICS")
print("-" * 65)

print(f"Daily volatility:             {std:>12.4%}")
print(f"Annualized volatility:        {annualized_std:>12.2%}")

print("\nDRAWDOWN METRICS")
print("-" * 65)

print(f"Worst daily return:           {worst_daily_return:>12.2%}")
print(f"Maximum drawdown:             {maximum_drawdown:>12.2%}")

print("\nDOWNSIDE RISK METRICS (1-DAY HORIZON)")
print("-" * 65)

print(f"{'Metric':<35}{'95%':>12}{'99%':>12}")
print("-" * 65)

print(
    f"{'Parametric VaR':<35}"
    f"{parametric_var_95:>12.2%}"
    f"{parametric_var_99:>12.2%}"
)

print(
    f"{'Historical VaR':<35}"
    f"{historical_var_95:>12.2%}"
    f"{historical_var_99:>12.2%}"
)

print(
    f"{'Historical Expected Shortfall':<35}"
    f"{expected_shortfall_95:>12.2%}"
    f"{expected_shortfall_99:>12.2%}"
)

print("\nRISK-ADJUSTED PERFORMANCE")
print("-" * 65)

print(f"Daily mean excess return:     {excess_return_mean:>12.4%}")
print(f"Daily excess return std:      {excess_return_std:>12.4%}")
print(f"Daily Sharpe ratio:           {sharpe_ratio:>12.4f}")
print(f"Annualized Sharpe ratio:      {annualized_sharpe:>12.4f}")

print("\nSAMPLE INFORMATION")
print("-" * 65)

print(f"Initial deposit date:         {transactions['date'].min():%Y-%m-%d}")
print(f"First valuation date:         {portfolio_wealth.index.min():%Y-%m-%d}")
print(f"First return date:            {portfolio_returns.index.min():%Y-%m-%d}")
print(f"Last return date:             {portfolio_returns.index.max():%Y-%m-%d}")
print(f"Daily return observations:    {len(portfolio_returns):>12}")
print(f"Excess return observations:   {len(excess_return):>12}")

print("\n" + "=" * 65)

          PERSONAL PORTFOLIO PERFORMANCE REPORT

PORTFOLIO VALUATION
-----------------------------------------------------------------
Initial investment:          $    1,067.00
Final securities value:      $      986.09
Accumulated dividends:       $        1.68
Total portfolio wealth:      $      987.77
Net profit/loss:             $      -79.23

RETURN METRICS
-----------------------------------------------------------------
Holding-period return:              -7.43%
Daily arithmetic mean:            -0.1192%
Daily geometric mean:             -0.1289%
Annualized arithmetic mean:        -30.03%

VOLATILITY METRICS
-----------------------------------------------------------------
Daily volatility:                  1.4072%
Annualized volatility:              22.34%

DRAWDOWN METRICS
-----------------------------------------------------------------
Worst daily return:                 -4.33%
Maximum drawdown:                  -12.89%

DOWNSIDE RISK METRICS (1-DAY HORIZON)
---------------

## 3.2. How has my portfolio performed relative to the market?

## 3.3. What are the stat characteristicis of my approach at the time I constructed the portfolio?